In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_date AS
WITH distinct_date AS (
    SELECT DISTINCT date 
    FROM bakery_pos_develop.silver.account_move_line
    WHERE date IS NOT NULL
)
SELECT 
    CAST(date_format(date, 'yyyyMMdd') AS INT) AS date_sk,
    date AS full_date,
    day(date) AS day,
    month(date) AS month,
    year(date) AS year
FROM distinct_date;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.gold.dim_date (
    date_sk INT NOT NULL,
    full_date DATE,
    day INT,
    month INT,
    year INT,
    CONSTRAINT dim_date_pk PRIMARY KEY (date_sk)
) USING DELTA;

In [0]:
%sql

MERGE INTO bakery_pos_develop.gold.dim_date AS target
USING (
    SELECT * FROM stg_date
) AS source
ON target.date_sk = source.date_sk
WHEN NOT MATCHED THEN
    INSERT (
        date_sk, 
        full_date, 
        day, 
        month, 
        year
    )
    VALUES (
        source.date_sk, 
        source.full_date, 
        source.day, 
        source.month, 
        source.year
    );

In [0]:
%sql

SELECT * FROM bakery_pos_develop.gold.dim_date